In [27]:
import numpy as np
from scipy.stats import chi2, norm

np.random.seed(42)
n = 100
alpha = 0.05

def merge_small_frequencies(obs, exp, edges):
    """Объединяет соседние интервалы, если эмпирическая частота < 5."""
    i = 0
    while i < len(obs) - 1:
        if obs[i] < 5:
            # Складываем с правым соседом
            obs[i] += obs[i+1]
            exp[i] += exp[i+1]
            # Удаляем правый сосед
            obs = np.delete(obs, i+1)
            exp = np.delete(exp, i+1)
            edges = np.delete(edges, i+1)
            # Не увеличиваем i, проверяем объединённый интервал снова
        else:
            i += 1
    # Проверка последнего интервала
    if len(obs) > 1 and obs[-1] < 5:
        obs[-2] += obs[-1]
        exp[-2] += exp[-1]
        obs = np.delete(obs, -1)
        exp = np.delete(exp, -1)
        edges = np.delete(edges, -1)
    return obs, exp, edges

print("\nЭкспоненциальное распределение (метод обратной функции)")

lambda_true = 0.1  # Параметр распределения
r = np.random.uniform(0, 1, n)
X_exp = - (1 / lambda_true) * np.log(r)

# 1. Статистические оценки и погрешности
mean_theo = 1 / lambda_true 
var_theo = 1 / lambda_true**2

mean_sample = np.mean(X_exp) # Выборочное среднее
var_sample = np.var(X_exp, ddof=1)  # Несмещённая оценка дисперсии

print(f"Теоретические значения: M(X) = {mean_theo:.4f}, D(X) = {var_theo:.4f}")
print(f"Выборочные оценки:      M(X) = {mean_sample:.4f}, D(X) = {var_sample:.4f}")
print(f"Абс. погрешности:       |M| = {abs(mean_sample - mean_theo):.4f}, |D| = {abs(var_sample - var_theo):.4f}")

# 2. Критерий Пирсона
k = int(round(1 + 3.3221 * np.log10(n))) 
h = (X_exp.max() - X_exp.min()) / k
edges = np.linspace(X_exp.min(), X_exp.max(), k + 1)
obs_freq, _ = np.histogram(X_exp, bins=edges) # Наблюдаемые частоты

lambda_hat = 1 / mean_sample
P_theo = np.exp(-lambda_hat * edges[:-1]) - np.exp(-lambda_hat * edges[1:])
exp_freq = n * P_theo # Теоретические частоты

print(f"\nДо объединения. \nНаблюдаемые частоты: {obs_freq} \nТеоретические частоты: {exp_freq}")

# Объединение малочисленных частот
obs_m, exp_m, edges_m = merge_small_frequencies(obs_freq.copy(), exp_freq.copy(), edges.copy())
k_merged = len(obs_m)

print(f"\nПосле объединения. \nНаблюдаемые частоты: {obs_m} \nТеоретические частоты: {exp_m}")

chi2_obs = np.sum((obs_m - exp_m)**2 / exp_m)
df = k_merged - 2  # Число степеней свободы для экспоненциального распределения
chi2_crit = chi2.ppf(1 - alpha, df)

print(f"\nКритерий Пирсона:")
print(f"Число интервалов: исходное = {k}, после объединения = {k_merged}")
print(f"hi^2_набл = {chi2_obs:.4f}")
print(f"hi^2_кр(α={alpha}, df={df}) = {chi2_crit:.4f}")
print(f"Вывод: {'Гипотеза о показательном распределении НЕ ОТВЕРГАЕТСЯ' if chi2_obs < chi2_crit else 'Гипотеза ОТВЕРГАЕТСЯ'}")




Экспоненциальное распределение (метод обратной функции)
Теоретические значения: M(X) = 10.0000, D(X) = 100.0000
Выборочные оценки:      M(X) = 10.9660, D(X) = 103.0354
Абс. погрешности:       |M| = 0.9660, |D| = 3.0354

До объединения. 
Наблюдаемые частоты: [46 24 11 10  5  3  0  1] 
Теоретические частоты: [44.09540501 24.41589512 13.51923028  7.48568039  4.14486696  2.29503816
  1.27077665  0.70363679]

После объединения. 
Наблюдаемые частоты: [46 24 11 10  9] 
Теоретические частоты: [44.09540501 24.41589512 13.51923028  7.48568039  8.41431856]

Критерий Пирсона:
Число интервалов: исходное = 8, после объединения = 5
hi^2_набл = 1.4441
hi^2_кр(α=0.05, df=3) = 7.8147
Вывод: Гипотеза о показательном распределении НЕ ОТВЕРГАЕТСЯ


In [ ]:

print("\nНормальное распределение")
mu_true, sigma_true = 3.0, 0.25

# Метод 1: Центральная предельная теорема (сумма 12 равномерных СВ)
r = np.random.uniform(0, 1, (n, 12)) # Матрица 100 x 12
z = np.sum(r, axis=1) - 6
X = mu_true + sigma_true * z

mean_clt = np.mean(X) # Выборочное среднее
var_clt = np.var(X, ddof=1) # Выборочная несмещенная дисперсия
std_clt = np.sqrt(var_clt)

print(f"\nВыборочное среднее = {mean_clt}")
print(f"Выборочная дисперсия = {var_clt}")
print(f"Выборочное среднее кв.откл. = {std_clt}")

abs_error_mean_clt = abs(mean_clt - mu_true)
abs_error_var_clt = abs(var_clt - sigma_true**2)
abs_error_std_clt = abs(std_clt - sigma_true)

rel_error_mean = (abs_error_mean_clt / mu_true) * 100
rel_error_std = (abs_error_std_clt / sigma_true) * 100

print(f"\nПогрешность мат. ожидания = {abs_error_mean_clt}")
print(f"Погрешность дисперсии = {abs_error_var_clt}")
print(f"Погрешность ср. кв. отклонения: = {abs_error_std_clt}")

print(f"\nОтносительная погрешность мат. ожидания = {rel_error_mean}%")
print(f"Относительная погрешность ср. кв. отклонения = {rel_error_std}%")


Нормальное распределение

Выборочное среднее = 2.9960061990546283
Выборочная дисперсия = 0.053637198432759096
Выборочное среднее кв.откл. = 0.23159706050111925

Погрешность мат. ожидания = 0.003993800945371717
Погрешность дисперсии = 0.008862801567240904
Погрешность ср. кв. отклонения: = 0.01840293949888075

Относительная погрешность мат. ожидания = 0.13312669817905726%
Относительная погрешность ср. кв. отклонения = 7.361175799552299%


In [ ]:
# Метод 2: Метод Мюллера
r1, r2 = np.random.uniform(0, 1, n), np.random.uniform(0, 1, n)
z = np.sqrt(-2 * np.log(r1)) * np.cos(2 * np.pi * r2)
X_muller = mu_true + sigma_true * z

mean_est = np.mean(X) # Выборочное среднее
var_est = np.var(X, ddof=1) # Выборочная несмещенная дисперсия
std_est = np.std(X, ddof=1)

print("\nПроверка гипотез о нормальном распределении")
